In [206]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [207]:
from ipm_newer.ipm_solve_no_split import ipm_solve_no_split # type: ignore[reportMissingImports]
from relaxers.remap_lower_bounds import remap_lower_bounds # type: ignore[reportMissingImports]
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [208]:
LIN_APPROX = 10

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = False

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

f_name = "GasLib-24_reform.json"

In [209]:
quad, q_time, q_x = solve_qcp_return_x(f_name)
print(quad)

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 49 columns and 0 nonzeros (Min)
Model fingerprint: 0x5b244183
Model has 21 linear objective coefficients
Model has 73 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [2e-09, 9e-03]
  QLMatrix range   [1e+00, 2e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [2e+02, 5e+03]
  RHS range        [0e+00, 0e+00]
  QRHS range       [5e+01, 4e+03]
Presolve removed 0 rows and 10 columns
Statistics for model 'qcp_from_json_pre':
  Problem type                : QCP
  Linear constraint matrix    : 65 rows, 65 columns, 132 nonzeros
  Variable types              : 65 continuous, 0 integer (0 binary)
  Quadratic constraints       : 13 constrs, 39 nonzeros
  Ma

In [210]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name=f_name)
remap_lower_bounds(f_name="linear_approx.json")
val, time, x, compl = ipm_solve_no_split()
print(x)

# print(condition_number_nes_basic())
print(f"Error: {val - quad}")

The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 55 iterations in 0.07 seconds.

Primal objective:        -4.07912355e+03
Dual objective:          -4.07912360e+03

Norm of Primal residual: 1.17e-12
Norm of Dual residual:   2.58e-12
Abs. Compl. Gap:         5.14e-05
Rel. Compl. Gap:         6.30e-09

[1.80560000e+02 1.00000000e+02 2.26614000e+02 5.44324000e+02 5.44324000e+02 2.47162000e+02 2.47162000e+02 2.97162000e+02 1.97162000e+02 4.44324000e+02 1.00000000e+02 3.44324000e+02 1.71368031e+02
 1.71368031e+02 1.72955969e+02 7.29559688e+01 1.00000000e+02 1.22162000e+02 1.22162000e+02 5.44324000e+02 3.44324000e+02 2.44324000e+02 1.03771000e+03 1.10000000e+03 2.26614000e+02 1.70894827e+03
 3.20000000e+03 1.70033789e+03 1.32105333e+02 8.25000000e+02 1.18658378e+03 1.18658378e+03 2.83347016e+02 1.70033789e+03 1.70033789e+03 2.88709212e-07 4.90784806e+02 3.25570771e+02 2.67642000e+02
 2.96606386e+02 2.86199671e+02 1.36564589e+02 2.527863

In [211]:
# Define a set of functions for each

quads = [[val] for val in x[:23]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            refinement_points = len(qs)
            uniform = np.linspace(lower, upper, num - refinement_points)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            return uniform
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name=f_name)
    remap_lower_bounds(f_name="linear_approx.json")

    val, time, x, compl = ipm_solve_no_split(f_name="linear_approx.json")

    # print(f"x: {x}")
    print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[j])

    print(quads)
    # print(compl)

# print(x)
# print(f"Error: {val - quad}")

Iteration 1
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 72 iterations in 0.09 seconds.

Primal objective:        -4.07314301e+03
Dual objective:          -4.07314303e+03

Norm of Primal residual: 2.70e-13
Norm of Dual residual:   4.68e-12
Abs. Compl. Gap:         1.38e-05
Rel. Compl. Gap:         1.69e-09

val: 4073.14302646392
Error: 0.20816577931782376
[[np.float64(180.56), np.float64(180.56)], [np.float64(100.0), np.float64(100.0)], [np.float64(226.61400000000017), np.float64(226.61400000000003)], [np.float64(544.3240000000001), np.float64(544.324)], [np.float64(544.3240000000001), np.float64(544.324)], [np.float64(247.16199999828132), np.float64(249.71217722260144)], [np.float64(247.16199999828132), np.float64(249.71217722260144)], [np.float64(297.16200000171887), np.float64(294.6118227773987)], [np.float64(197.16200000171878), np.float64(194.61182277739854)], [np.float64(444.32400000000007), np.float64(444.3239999999999